# 3. Reinforcement Learning with Verifiable Rewards (GRPO)
**HLT 2026 - Reasoning with Verifiable Rewards**

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using **Group Relative Policy Optimization (GRPO)** as introduced in **DeepSeekMath** and **DeepSeek-R1-Zero**:
1. **Zero Critic Architecture**: Eliminates PPO's auxiliary Value Model by generating $G$ completions per prompt and calculating relative group advantages ($A_i = \frac{r_i - \mu}{\sigma}$).
2. **Deterministic Verifiable Rewards**: 3 rule-based signals (Accuracy, Format, Validity).
3. **PEFT (LoRA)**: Parameter-efficient fine-tuning that fits comfortably inside free T4 GPU VRAM (15–16 GB) without OOM.

> ⚠️ **Hardware Requirement**: **Mandatory GPU** (free Colab T4 or Kaggle 2xT4). CPU training is computationally infeasible due to group rollout generation.

## 1. Install Dependencies (Colab / Kaggle)

In [ ]:
!pip install -q "trl>=0.14.0" "transformers>=4.48.0" peft accelerate datasets bitsandbytes

## 2. Hardware Verification & Model Configuration

In [ ]:
import os
import ast
import re
import json
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

assert torch.cuda.is_available(), "GPU not detected! Enable GPU via Runtime -> Change runtime type -> T4 GPU"
print(f"Active GPU: {torch.cuda.get_device_name(0)}")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

## 3. Load Training Dataset for GRPOTrainer

In [ ]:
candidates = [
    "data/train_countdown.jsonl",
    "../data/train_countdown.jsonl",
    "/content/data/train_countdown.jsonl"
]
train_file = next((p for p in candidates if os.path.exists(p)), None)

if train_file:
    with open(train_file, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]
    print(f"Loaded {len(data)} training samples from {train_file}")
else:
    print("Training dataset not found locally. Using fallback samples...")
    data = [
        {"numbers": [84, 83, 34, 72], "target": 39},
        {"numbers": [25, 5, 2, 8], "target": 60},
        {"numbers": [12, 3, 4, 7], "target": 35},
        {"numbers": [50, 2, 3, 5], "target": 105}
    ] * 250

train_prompts = []
for d in data:
    train_prompts.append({
        "prompt": [
            {"role": "system", "content": "You are a helpful mathematical assistant."},
            {"role": "user", "content": f"Using the numbers {d['numbers']}, create an equation that equals {d['target']}. You can use basic arithmetic operations (+, -, *, /) and each number can be used at most once. Show your reasoning in <think>...</think> and place the final equation in <answer>...</answer>."}
        ],
        "numbers": d["numbers"],
        "target": d["target"]
    })

train_dataset = Dataset.from_list(train_prompts)
print(f"Dataset ready: {len(train_dataset)} instances for RL training.")

## 4. Define Verifiable Reward Functions (R1-Zero Paradigm)

In [ ]:
ALLOWED_OPERATORS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b if b != 0 else None,
    ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
    ast.USub: lambda a: -a,
    ast.UAdd: lambda a: +a
}

ALLOWED_NODE_TYPES = (
    ast.Expression,
    ast.BinOp,
    ast.UnaryOp,
    ast.Constant,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.USub,
    ast.UAdd
)

def is_safe_math_ast(tree):
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODE_TYPES):
            return False
        if isinstance(node, ast.BinOp) and type(node.op) not in ALLOWED_OPERATORS:
            return False
        if isinstance(node, ast.UnaryOp) and type(node.op) not in ALLOWED_OPERATORS:
            return False
        if isinstance(node, ast.Constant):
            if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                return False
    return True

def format_reward_func(completions, **kwargs):
    """Rewards (+0.2) compliance with <think>...</think><answer>...</answer> structure."""
    rewards = []
    pattern = r"^<think>.*?</think>s*<answer>.*?</answer>$"
    for comp in completions:
        content = comp[0]["content"] if isinstance(comp, list) else comp
        if re.search(pattern, content.strip(), re.DOTALL | re.IGNORECASE):
            rewards.append(0.2)
        elif "<answer>" in content and "</answer>" in content:
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

def equation_validity_reward_func(completions, numbers, **kwargs):
    """Rewards (+0.1) valid syntax and adherence to provided numbers."""
    rewards = []
    for comp, nums in zip(completions, numbers):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        ans_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if not ans_match:
            rewards.append(0.0)
            continue
        expr = ans_match.group(1).split("=")[0].strip()
        try:
            tree = ast.parse(expr, mode='eval')
            if not is_safe_math_ast(tree):
                rewards.append(0.0)
                continue
            used = []
            for n in ast.walk(tree):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                    used.append(int(n.value) if float(n.value).is_integer() else n.value)
            avail = list(nums)
            valid = True
            for u in used:
                if u in avail:
                    avail.remove(u)
                else:
                    valid = False
                    break
            rewards.append(0.1 if (valid and len(used) > 0) else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def accuracy_reward_func(completions, numbers, target, **kwargs):
    """Rewards (+1.0) exact mathematical equality with the target number."""
    rewards = []
    for comp, nums, tgt in zip(completions, numbers, target):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        ans_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if not ans_match:
            rewards.append(0.0)
            continue
        expr = ans_match.group(1).split("=")[0].strip()
        try:
            tree = ast.parse(expr, mode='eval')
            if not is_safe_math_ast(tree):
                rewards.append(0.0)
                continue
            used = []
            for n in ast.walk(tree):
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                    used.append(int(n.value) if float(n.value).is_integer() else n.value)
            avail = list(nums)
            valid = True
            for u in used:
                if u in avail:
                    avail.remove(u)
                else:
                    valid = False
                    break
            if not valid or len(used) == 0:
                rewards.append(0.0)
                continue

            def ev(node):
                if isinstance(node, ast.Expression):
                    return ev(node.body)
                elif isinstance(node, ast.Constant):
                    return float(node.value)
                elif isinstance(node, ast.UnaryOp):
                    op = ALLOWED_OPERATORS.get(type(node.op))
                    v = ev(node.operand)
                    return op(v) if op and v is not None else None
                elif isinstance(node, ast.BinOp):
                    op = ALLOWED_OPERATORS.get(type(node.op))
                    l = ev(node.left)
                    r = ev(node.right)
                    return op(l, r) if op and l is not None and r is not None else None
                return None

            computed = ev(tree)
            if computed is not None and abs(computed - tgt) < 1e-6:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

## 5. Configure LoRA (PEFT) & GRPOTrainer

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

models_dir = "models" if not os.path.basename(os.getcwd()) == "notebooks" else "../models"
os.makedirs(models_dir, exist_ok=True)

training_args = GRPOConfig(
    output_dir=os.path.join(models_dir, "qwen-countdown-grpo"),
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4, # Group size G=4 for relative reward whitening
    max_completion_length=384,
    max_prompt_length=256,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none"
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=[accuracy_reward_func, format_reward_func, equation_validity_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)

## 6. Launch GRPO Training

In [ ]:
print("🚀 Launching GRPO RL training with verifiable rewards...")
trainer.train()

final_save_path = os.path.join(models_dir, "qwen-countdown-grpo-final")
trainer.model.save_pretrained(final_save_path)
print(f"✅ Model trained successfully! LoRA weights saved to {final_save_path}")

## 7. Final 3-Way Comparative Evaluation (For University Report)

In [ ]:
res_candidates = [
    "results/baselines.json",
    "../results/baselines.json",
    "/content/results/baselines.json"
]
base_path = next((p for p in res_candidates if os.path.exists(p)), None)

if base_path:
    with open(base_path) as f:
        base = json.load(f)
    print("=== PROJECT COMPARATIVE SUMMARY TABLE ===")
    print(f"1. Zero-Shot Direct:        {base['zero_shot_accuracy']:.2f}% (Avg {base['zero_shot_tokens']:.1f} tokens)")
    print(f"2. Chain-of-Thought (CoT):  {base['cot_accuracy']:.2f}% (Avg {base['cot_tokens']:.1f} tokens)")
    print("3. GRPO (RL Model):          [Evaluated on test_countdown.jsonl with trained policy]")